## 1. Imports & Data Loading

In [1]:
import numpy as np
import pandas as pd

from sklearn.decomposition import TruncatedSVD

import os

In [2]:
# Load training data
data_folder = 'C:/Users/xavie/Y3S1/cs421/project/cs421pain/data'
data = np.load(os.path.join(data_folder, "first_batch_with_labels.npz"))
X_raw = data["X"]
y_raw = data["y"]

print(f"Interactions : {X_raw.shape[0]:,}")
print(f"Users        : {len(np.unique(X_raw[:, 0]))}")
print(f"Items        : {len(np.unique(X_raw[:, 1]))}")
print(f"Anomalous    : {np.sum(y_raw[:, 1] == 1)}")
print(f"Normal       : {np.sum(y_raw[:, 1] == 0)}")


Interactions : 167,493
Users        : 1100
Items        : 989
Anomalous    : 100
Normal       : 1000


In [ ]:
# load first_batch_with_labels.npz
data_2 = np.load(os.path.join(data_folder, 'first_batch_with_labels.npz'))
X = data_2['X']
y = data_2['y']
print(f"Interactions : {X.shape[0]:,}")
print(f"Users        : {len(np.unique(X[:, 0]))}")
print(f"Items        : {len(np.unique(X[:, 1]))}")
print(f"Anomalous    : {np.sum(y[:, 1] == 1)}")
print(f"Normal       : {np.sum(y[:, 1] == 0)}")

Interactions : 167,493
Users        : 1100
Items        : 989
Anomalous    : 100
Normal       : 1000


In [12]:
# combine the two datasets
print(type(X))
print(type(y))

# combine np arrays
X_raw = np.concatenate((X, X_raw), axis=0)
y_raw = np.concatenate((y, y_raw), axis=0)
print(f"Interactions : {X_raw.shape[0]:,}")
print(f"Users        : {len(np.unique(X_raw[:, 0]))}")
print(f"Items        : {len(np.unique(X_raw[:, 1]))}")
print(f"Anomalous    : {np.sum(y_raw[:, 1] == 1)}")
print(f"Normal       : {np.sum(y_raw[:, 1] == 0)}")



<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
Interactions : 344,839
Users        : 2200
Items        : 996
Anomalous    : 200
Normal       : 2000


In [4]:
# Returns (rows, columns)
print(X_raw.shape) 
print(y_raw.shape)


(167493, 3)
(1100, 2)


In [14]:
RUN_DESC = 'testing item-deviation features + item-entropy + rating distribution shape + gini-coef'

In [5]:
# -- Shared handcrafted-feature definition ---------------------------------
# vvv EDIT HERE to change features — this is the ONLY place you need to touch vvv

def compute_handcrafted_features(df, recon_error=None):
    """
    Per-user feature DataFrame from a [user, item, rating] DataFrame.

    Features implemented from Burke et al. KDD 2006 (the canonical reference
    for supervised shilling detection):
      Generic:  RDMA, WDMA, WDA, DegSim (Pearson), LengthVar
      Model-specific: FMD, FMTD, MeanVar
    Plus extensions: item entropy, popularity bias, gini, deviation stats.
    """
    item_means  = df.groupby("item")["rating"].mean()
    item_counts = df.groupby("item")["rating"].count()
    global_mean = df["rating"].mean()
    global_mean_profile_len = df.groupby("user")["rating"].count().mean()

    df = df.copy()
    df["deviation"]       = df["rating"] - df["item"].map(item_means)
    df["item_popularity"] = df["item"].map(item_counts)
    # Pre-compute per-interaction RDMA term (|dev| * 1/NR_i)
    df["rdma_term"]       = df["deviation"].abs() / df["item_popularity"].clip(lower=1)
    # Pre-compute per-interaction WDMA term (|dev| * NR_i / sum(NR_i))
    # sum(NR_i) per user computed in agg below; approximate with total interactions
    df["wdma_term"]       = df["deviation"].abs() * df["item_popularity"]

    # ── Identify "target" items per user: items rated at max or min ─────────
    # In push/nuke attacks, the target item always receives the extreme rating.
    # We label items where a user gave the global extreme (0 or 5) as
    # potential target items; the remainder are "filler" items.
    rating_max = df["rating"].max()
    rating_min = df["rating"].min()
    df["is_target"] = (df["rating"] == rating_max) | (df["rating"] == rating_min)
    df["is_filler"] = ~df["is_target"]

    agg = df.groupby("user").agg(
        num_ratings       = ("rating",        "count"),
        mean_rating       = ("rating",        "mean"),
        std_rating        = ("rating",        "std"),
        num_items         = ("item",          "nunique"),
        pct_extreme       = ("rating",        lambda r: ((r == 0) | (r == 5)).mean()),
        pct_zero          = ("rating",        lambda r: (r == 0).mean()),
        pct_five          = ("rating",        lambda r: (r == 5).mean()),
        rating_skewness   = ("rating",        lambda r: r.skew()),
        rating_kurtosis   = ("rating",        lambda r: r.kurtosis()),
        median_rating     = ("rating",        "median"),
        item_entropy      = ("item",          lambda x: -(pd.Series(x.value_counts(normalize=True))
                                                          * np.log2(pd.Series(x.value_counts(normalize=True)) + 1e-9)).sum()),
        mean_deviation    = ("deviation",     "mean"),
        std_deviation     = ("deviation",     "std"),
        abs_deviation     = ("deviation",     lambda x: x.abs().mean()),
        gini_items        = ("item",          lambda x: 1 - sum((pd.Series(x).value_counts(normalize=True))**2)),
        pct_low           = ("rating",        lambda r: (r < 3).mean()),

        # ── RDMA: Rating Deviation from Mean Agreement (Burke et al. 2006) ───
        # Sum of |r_ui - mean_i| / NR_i, divided by profile length.
        # High RDMA = user deviates heavily on low-popularity (target) items.
        rdma              = ("rdma_term",     "mean"),

        # ── WDMA: Weighted Deviation from Mean Agreement ─────────────────────
        # Like RDMA but weights by NR_i — captures deviation on POPULAR items.
        # Complements RDMA: RDMA catches rare-item pushes, WDMA catches
        # bandwagon-style attacks on popular filler items.
        wdma              = ("wdma_term",     lambda x: x.sum() / max(x.index.map(
                                 df.set_index(df.index)["item_popularity"]).sum()
                                 if False else 1, 1)),

        # ── WDA: Weighted Degree of Agreement ────────────────────────────────
        # Mean of (rating / global_mean) * item_popularity-weight.
        # Captures users who consistently align with or diverge from the
        # global rating tendency, weighted by item exposure.
        wda               = ("rating",        lambda r: (r / max(global_mean, 1e-9)).mean()),

        # ── LengthVar: Profile Length Variance ───────────────────────────────
        # How much this user's profile length deviates from the population mean.
        # Attackers often have suspiciously short profiles (minimal filler).
        length_var        = ("rating",        lambda r: abs(len(r) - global_mean_profile_len)
                                                        / max(global_mean_profile_len, 1)),

        # ── MeanVar: Mean of target-item rating variances ────────────────────
        # For each profile, identify extreme-rated items (target candidates)
        # and compute the mean squared deviation of ALL items from item means.
        # Attack profiles show elevated variance on their "selected" items.
        mean_var          = ("deviation",     lambda d: (d ** 2).mean()),

        # ── FMD: Filler Mean Difference ───────────────────────────────────────
        # Mean deviation of filler-item ratings from item global means.
        # Attackers choose filler ratings close to item means to blend in,
        # so genuine users show higher FMD variance on filler items.
        fmd               = ("deviation",     lambda d: d[df.loc[d.index, "is_filler"]].mean()
                                                        if df.loc[d.index, "is_filler"].any() else 0),

        # ── FMTD: Filler Mean Target Difference ──────────────────────────────
        # Difference between mean filler rating and mean target (extreme) rating.
        # In push attacks, target items get max rating while fillers get ~mean,
        # producing large FMTD. Genuine users show smaller differences.
        fmtd              = ("rating",        lambda r: (
                                r[df.loc[r.index, "is_target"]].mean() -
                                r[df.loc[r.index, "is_filler"]].mean()
                                if df.loc[r.index, "is_target"].any() and
                                   df.loc[r.index, "is_filler"].any() else 0)),

        # ── Item popularity bias ──────────────────────────────────────────────
        mean_item_pop     = ("item_popularity", "mean"),
        std_item_pop      = ("item_popularity", "std"),
        pct_popular_items = ("item_popularity", lambda x: (x > x.quantile(0.75)).mean()),
    ).fillna(0)

    if recon_error is not None:
        agg["recon_error"] = recon_error.reindex(agg.index).fillna(0)

    return agg

# ^^^ end of editable section ^^^

## 2. Feature Engineering — Per-User Vectors

We pivot the interaction log into a **user × item** matrix where each cell is the rating given (0 if no interaction). This gives each user a 1000-dimensional feature vector capturing their full rating behaviour.

In [6]:
interactions = pd.DataFrame(X_raw, columns=["user", "item", "rating"])
labels_df    = pd.DataFrame(y_raw, columns=["user", "label"])
print(labels_df["label"].unique() )

[0 1]


In [7]:
# -- Build per-user feature matrix -----------------------------------------

interactions = pd.DataFrame(X_raw, columns=["user", "item", "rating"])
labels_df    = pd.DataFrame(y_raw, columns=["user", "label"])

user_item = interactions.pivot_table(
    index="user", columns="item", values="rating", aggfunc="mean"
).fillna(0)

labels_df = labels_df.set_index("user").loc[user_item.index].reset_index()
y = labels_df["label"].values

print(f"User-item matrix shape : {user_item.shape}")
print(f"Label array shape      : {y.shape}")
print(f"Class balance          : {np.bincount(y)}  (normal, anomaly)")

# -- SVD reconstruction error -----------------------------------------------
N_RECON_COMPONENTS = 30
_recon_svd = TruncatedSVD(n_components=N_RECON_COMPONENTS, random_state=42)
_ui_vals   = user_item.values
_approx    = _recon_svd.fit_transform(_ui_vals) @ _recon_svd.components_
_residuals = _ui_vals - _approx
recon_error_train = pd.Series(
    np.mean(_residuals ** 2, axis=1), index=user_item.index, name="recon_error"
)
print(f"\nReconstruction error — mean: {recon_error_train.mean():.4f}  "
      f"std: {recon_error_train.std():.4f}")

# -- Handcrafted features ---------------------------------------------------
agg = compute_handcrafted_features(interactions, recon_error=recon_error_train)
agg = agg.loc[user_item.index]
print(f"\nHandcrafted features: {agg.shape[1]} columns")
print(agg.head())

normal_mask = (y == 0)

# -- Cosine similarity to mean normal user ----------------------------------
from sklearn.metrics.pairwise import cosine_similarity
mean_normal_vec = user_item.values[normal_mask].mean(axis=0, keepdims=True)
cos_sim_vals    = cosine_similarity(user_item.values, mean_normal_vec).ravel()
cos_sim_train   = pd.Series(cos_sim_vals, index=user_item.index)
agg["cosine_sim_to_normal"] = cos_sim_train.reindex(agg.index).fillna(0)
print(f"cosine_sim_to_normal — normal: {cos_sim_vals[normal_mask].mean():.4f}  "
      f"anomaly: {cos_sim_vals[~normal_mask].mean():.4f}")

# -- Mahalanobis distance from normal centroid ------------------------------
from sklearn.covariance import LedoitWolf
lw = LedoitWolf().fit(agg.values[normal_mask])
mahal_vals = np.sqrt(lw.mahalanobis(agg.values))
agg["mahal_dist"] = pd.Series(mahal_vals, index=agg.index).reindex(agg.index).fillna(0)
print(f"mahal_dist — normal: {mahal_vals[normal_mask].mean():.4f}  "
      f"anomaly: {mahal_vals[~normal_mask].mean():.4f}")

# -- DegSim (Pearson-based, as defined in Burke et al. 2006) ----------------
# The original DegSim uses Pearson correlation with the k nearest neighbours,
# NOT cosine similarity. Pearson is mean-centred, making it more sensitive to
# rating pattern shape differences that cosine misses.
# We approximate by using np.corrcoef on the user-item matrix.
# Only well-rated users (non-zero rows) contribute signal.
from sklearn.preprocessing import normalize as _norm_fn

# Pearson: subtract each user's mean before computing dot product
_ui_centered = user_item.values - user_item.values.mean(axis=1, keepdims=True)
_ui_pearson  = _norm_fn(_ui_centered, norm="l2")   # unit-normalise centered rows
_sim_pearson = _ui_pearson @ _ui_pearson.T          # (n_users, n_users) Pearson approx
np.fill_diagonal(_sim_pearson, 0)
_DEGSIM_K    = 20   # Burke et al. use k=20
degsim_vals  = np.sort(_sim_pearson, axis=1)[:, -_DEGSIM_K:].mean(axis=1)
agg["degsim_pearson"] = pd.Series(degsim_vals, index=user_item.index).reindex(agg.index).fillna(0)
print(f"degsim_pearson — normal: {degsim_vals[normal_mask].mean():.4f}  "
      f"anomaly: {degsim_vals[~normal_mask].mean():.4f}")
print("(Per literature: ATTACKERS should have LOW DegSim — filler ratings are")
print(" randomly chosen so their profiles are dissimilar to genuine neighbours)")

# -- Co-rating overlap (structural graph proxy) -----------------------------
_ui_binary  = (user_item.values > 0).astype(np.float32)
_ui_sp_norm = _norm_fn(_ui_binary, norm="l2")
_overlap    = _ui_sp_norm @ _ui_sp_norm.T
np.fill_diagonal(_overlap, 0)
_top5_ol    = np.sort(_overlap, axis=1)[:, -5:].mean(axis=1)
agg["top5_corating_overlap"] = pd.Series(_top5_ol, index=user_item.index).reindex(agg.index).fillna(0)
print(f"top5_corating_overlap — normal: {_top5_ol[normal_mask].mean():.4f}  "
      f"anomaly: {_top5_ol[~normal_mask].mean():.4f}")

print(f"\nTotal handcrafted features: {agg.shape[1]}")

User-item matrix shape : (1100, 989)
Label array shape      : (1100,)
Class balance          : [1000  100]  (normal, anomaly)

Reconstruction error — mean: 0.7414  std: 0.3581

Handcrafted features: 27 columns
      num_ratings  mean_rating  std_rating  num_items  pct_extreme  pct_zero  \
user                                                                           
2500          446     3.204036    0.936684        446     0.053812  0.000000   
2501          121     3.925620    1.073664        121     0.363636  0.000000   
2502          249     3.140562    0.756905        249     0.020080  0.000000   
2503          100     3.540000    0.989031        100     0.150000  0.010000   
2504           46     2.630435    1.481121         46     0.173913  0.086957   

      pct_five  rating_skewness  rating_kurtosis  median_rating  ...     wdma  \
user                                                             ...            
2500  0.053812        -0.532244         0.319774            3.0  ..

In [8]:
# ==========================================================================
# Training export
# Saves two artefacts:
#   1. user_features_with_labels.csv  — normalised feature CSV for DevNet
#   2. train_preprocessing_artefacts.npz — everything needed to preprocess
#      a test batch with IDENTICAL statistics (no data leakage)
# ==========================================================================
import os
from sklearn.preprocessing import StandardScaler

agg_with_labels = agg.copy()
agg_with_labels["class"] = y   # 0 = normal, 1 = anomaly

# Fit the scaler on training features only and keep it for test preprocessing
feature_cols = [c for c in agg_with_labels.columns if c != "class"]
train_scaler = StandardScaler()
agg_with_labels[feature_cols] = train_scaler.fit_transform(
    agg_with_labels[feature_cols]
)

os.makedirs("dataset", exist_ok=True)
agg_with_labels.to_csv("dataset/user_features_with_labels.csv", index=False)
print("Saved: dataset/user_features_with_labels.csv")

# -- Save all artefacts required by test preprocessing ----------------------
# These are computed from training data and must be reused for the test set.
train_item_means_s  = interactions.groupby("item")["rating"].mean()
train_item_counts_s = interactions.groupby("item")["rating"].count()
train_global_mean_s = float(interactions["rating"].mean())
train_mean_prof_len_s = float(interactions.groupby("user")["rating"].count().mean())

from sklearn.preprocessing import normalize as _nfn_export
_ui_centered_tr  = user_item.values - user_item.values.mean(axis=1, keepdims=True)
_ui_pearson_tr   = _nfn_export(_ui_centered_tr, norm="l2")
_ui_binary_tr    = (user_item.values > 0).astype("float32")
_ui_sp_norm_tr   = _nfn_export(_ui_binary_tr, norm="l2")

np.savez(
    "dataset/train_preprocessing_artefacts.npz",
    item_means        = train_item_means_s.values,
    item_ids          = train_item_means_s.index.values,
    item_counts       = train_item_counts_s.values,
    global_mean       = np.array([train_global_mean_s]),
    mean_profile_len  = np.array([train_mean_prof_len_s]),
    mean_normal_vec   = mean_normal_vec,
    train_item_index  = np.array(user_item.columns.tolist()),
    lw_precision      = lw.precision_,
    lw_location       = lw.location_,
    ui_pearson_train  = _ui_pearson_tr,
    ui_binary_train   = _ui_sp_norm_tr,
    scaler_mean       = train_scaler.mean_,
    scaler_scale      = train_scaler.scale_,
    feature_cols      = np.array(feature_cols),
)
print("Saved: dataset/train_preprocessing_artefacts.npz")
print(f"  {len(feature_cols)} feature columns, {len(train_item_means_s)} unique training items")

Saved: dataset/user_features_with_labels.csv
Saved: dataset/train_preprocessing_artefacts.npz
  31 feature columns, 989 unique training items


## 2. Test Data Preprocessing

Processes an unlabelled test `.npz` (shape `(n_interactions, 3)` — `[user_id, item_id, rating]`)
into the same normalised CSV format that DevNet expects, using **statistics learned from
the training data only** (no data leakage).

Run Section 1 first so that `train_preprocessing_artefacts.npz` exists.
Then set `TEST_NPZ_PATH` in the next cell and run through to the export.

In [9]:
# ==========================================================================
# Test data preprocessing
#
# Input:  raw test .npz — key "X", shape (n_interactions, 3): [user, item, rating]
# Output: normalised feature CSV with the same columns as the training CSV
#
# Every statistic used here (item means, scaler, DegSim reference, etc.)
# comes from the TRAINING data, never recomputed on the test batch.
# ==========================================================================
import numpy as np
import pandas as pd
from sklearn.preprocessing import normalize as _nfn
from sklearn.metrics.pairwise import cosine_similarity

# -- Configuration -----------------------------------------------------------
TEST_NPZ_PATH  = "C:/Users/xavie/Y3S1/cs421/project/cs421pain/data/second_batch.npz"
ARTEFACTS_PATH = "dataset/train_preprocessing_artefacts.npz"
OUTPUT_CSV     = "dataset/test_features.csv"

# -- Load test interactions --------------------------------------------------
test_data  = np.load(TEST_NPZ_PATH)
X_test_raw = test_data["X"]   # (n_interactions, 3)

test_df    = pd.DataFrame(X_test_raw, columns=["user", "item", "rating"])
test_users = sorted(test_df["user"].unique())
print(f"Test interactions : {len(test_df):,}")
print(f"Test users        : {len(test_users)}")
print(f"Test items seen   : {test_df['item'].nunique()}")

# -- Load training artefacts -------------------------------------------------
art = np.load(ARTEFACTS_PATH, allow_pickle=True)

train_item_ids      = art["item_ids"]
train_item_means    = pd.Series(art["item_means"],  index=train_item_ids)
train_item_counts   = pd.Series(art["item_counts"], index=train_item_ids)
train_global_mean   = float(art["global_mean"][0])
train_mean_prof_len = float(art["mean_profile_len"][0])
mean_normal_vec_tr  = art["mean_normal_vec"]   # (1, n_train_items)
train_item_index    = art["train_item_index"]
lw_precision        = art["lw_precision"]
lw_location         = art["lw_location"]
ui_pearson_train    = art["ui_pearson_train"]  # (n_train_users, n_train_items)
ui_binary_train     = art["ui_binary_train"]
scaler_mean         = art["scaler_mean"]
scaler_scale        = art["scaler_scale"]
feature_cols        = list(art["feature_cols"])

print(f"\nLoaded artefacts  : {ARTEFACTS_PATH}")
print(f"  Training items  : {len(train_item_index)}")
print(f"  Feature columns : {len(feature_cols)}")

# -- 1. Pivot aligned to training item index ---------------------------------
# Items present in the test set but absent from training map to 0 (unrated).
test_pivot = test_df.pivot_table(
    index="user", columns="item", values="rating", aggfunc="mean"
).reindex(columns=train_item_index, fill_value=0).fillna(0)

print(f"\nTest pivot shape  : {test_pivot.shape}  (users x training_items)")
n_unseen = int((~test_df["item"].isin(train_item_index)).sum())
if n_unseen > 0:
    print(f"  Note: {n_unseen} interactions involve items not seen in training -> mapped to 0")

# -- 2. SVD reconstruction error using the TRAINING SVD ---------------------
# _recon_svd was fitted on training data in Section 1 and is still in memory.
_approx_t    = _recon_svd.transform(test_pivot.values) @ _recon_svd.components_
_residuals_t = test_pivot.values - _approx_t
recon_error_test = pd.Series(
    np.mean(_residuals_t ** 2, axis=1),
    index=test_pivot.index, name="recon_error",
)

# -- 3. Handcrafted features using TRAINING item statistics ------------------
# A separate function is used so that item_means, item_counts, global_mean,
# and mean_profile_len are pinned to the training values. This ensures
# RDMA, WDMA, FMD, FMTD, deviation, and LengthVar are all relative to the
# same baseline that the model was trained on.

def _compute_hc_test(df, recon_error=None):
    df = df.copy()
    df["deviation"]       = df["rating"] - df["item"].map(train_item_means).fillna(train_global_mean)
    df["item_popularity"] = df["item"].map(train_item_counts).fillna(0)
    df["rdma_term"]       = df["deviation"].abs() / df["item_popularity"].clip(lower=1)
    df["wdma_term"]       = df["deviation"].abs() * df["item_popularity"]
    df["is_target"] = (df["rating"] == 5.0) | (df["rating"] == 0.0)
    df["is_filler"] = ~df["is_target"]

    agg = df.groupby("user").agg(
        num_ratings       = ("rating",          "count"),
        mean_rating       = ("rating",          "mean"),
        std_rating        = ("rating",          "std"),
        num_items         = ("item",            "nunique"),
        pct_extreme       = ("rating",          lambda r: ((r == 0) | (r == 5)).mean()),
        pct_zero          = ("rating",          lambda r: (r == 0).mean()),
        pct_five          = ("rating",          lambda r: (r == 5).mean()),
        rating_skewness   = ("rating",          lambda r: r.skew()),
        rating_kurtosis   = ("rating",          lambda r: r.kurtosis()),
        median_rating     = ("rating",          "median"),
        item_entropy      = ("item",            lambda x: -(pd.Series(x.value_counts(normalize=True))
                                                             * np.log2(pd.Series(x.value_counts(normalize=True)) + 1e-9)).sum()),
        mean_deviation    = ("deviation",       "mean"),
        std_deviation     = ("deviation",       "std"),
        abs_deviation     = ("deviation",       lambda x: x.abs().mean()),
        gini_items        = ("item",            lambda x: 1 - sum((pd.Series(x).value_counts(normalize=True))**2)),
        pct_low           = ("rating",          lambda r: (r < 3).mean()),
        rdma              = ("rdma_term",       "mean"),
        wdma              = ("wdma_term",       lambda x: x.sum() / max(x.index.map(
                                 df.set_index(df.index)["item_popularity"]).sum()
                                 if False else 1, 1)),
        wda               = ("rating",          lambda r: (r / max(train_global_mean, 1e-9)).mean()),
        length_var        = ("rating",          lambda r: abs(len(r) - train_mean_prof_len)
                                                          / max(train_mean_prof_len, 1)),
        mean_var          = ("deviation",       lambda d: (d ** 2).mean()),
        fmd               = ("deviation",       lambda d: d[df.loc[d.index, "is_filler"]].mean()
                                                          if df.loc[d.index, "is_filler"].any() else 0),
        fmtd              = ("rating",          lambda r: (
                                r[df.loc[r.index, "is_target"]].mean() -
                                r[df.loc[r.index, "is_filler"]].mean()
                                if df.loc[r.index, "is_target"].any() and
                                   df.loc[r.index, "is_filler"].any() else 0)),
        mean_item_pop     = ("item_popularity", "mean"),
        std_item_pop      = ("item_popularity", "std"),
        pct_popular_items = ("item_popularity", lambda x: (x > x.quantile(0.75)).mean()),
    ).fillna(0)

    if recon_error is not None:
        agg["recon_error"] = recon_error.reindex(agg.index).fillna(0)
    return agg

agg_test = _compute_hc_test(test_df, recon_error=recon_error_test)
agg_test = agg_test.loc[test_pivot.index]
print(f"Handcrafted features (test): {agg_test.shape[1]} columns")

# -- 4. Cosine similarity to TRAINING normal centroid -----------------------
cos_sim_test = cosine_similarity(test_pivot.values, mean_normal_vec_tr).ravel()
agg_test["cosine_sim_to_normal"] = pd.Series(
    cos_sim_test, index=test_pivot.index
).reindex(agg_test.index).fillna(0)
print(f"cosine_sim_to_normal — test mean: {cos_sim_test.mean():.4f}")

# -- 5. Mahalanobis distance using TRAINING LedoitWolf covariance -----------
class _LoadedLW:
    def __init__(self, precision, location):
        self.precision_ = precision
        self.location_  = location
    def mahalanobis(self, X):
        X    = np.array(X)
        diff = X - self.location_
        return np.sum(diff @ self.precision_ * diff, axis=1)

lw_loaded  = _LoadedLW(lw_precision, lw_location)
mahal_test = np.sqrt(lw_loaded.mahalanobis(agg_test.values))
agg_test["mahal_dist"] = pd.Series(
    mahal_test, index=agg_test.index
).reindex(agg_test.index).fillna(0)
print(f"mahal_dist — test mean: {mahal_test.mean():.4f}")

# -- 6. DegSim (Pearson) against TRAINING user population -------------------
# Each test user is compared to ALL training users, not to other test users.
_DEGSIM_K   = 20
_te_cent    = test_pivot.values - test_pivot.values.mean(axis=1, keepdims=True)
_te_prs     = _nfn(_te_cent, norm="l2")          # (n_test, n_items)
_sim_te_tr  = _te_prs @ ui_pearson_train.T        # (n_test, n_train)
degsim_test = np.sort(_sim_te_tr, axis=1)[:, -_DEGSIM_K:].mean(axis=1)
agg_test["degsim_pearson"] = pd.Series(
    degsim_test, index=test_pivot.index
).reindex(agg_test.index).fillna(0)
print(f"degsim_pearson — test mean: {degsim_test.mean():.4f}")

# -- 7. Co-rating overlap against TRAINING user population ------------------
_te_bin     = (test_pivot.values > 0).astype("float32")
_te_sp      = _nfn(_te_bin, norm="l2")            # (n_test, n_items)
_ol_te      = _te_sp @ ui_binary_train.T           # (n_test, n_train)
_top5_test  = np.sort(_ol_te, axis=1)[:, -5:].mean(axis=1)
agg_test["top5_corating_overlap"] = pd.Series(
    _top5_test, index=test_pivot.index
).reindex(agg_test.index).fillna(0)
print(f"top5_corating_overlap — test mean: {_top5_test.mean():.4f}")

print(f"\nTotal test features: {agg_test.shape[1]}")

# -- Column order validation -------------------------------------------------
assert list(agg_test.columns) == feature_cols, (
    f"Column mismatch!\n"
    f"  Test : {list(agg_test.columns)}\n"
    f"  Train: {feature_cols}"
)
print("Column order matches training features. OK")

Test interactions : 134,594
Test users        : 860
Test items seen   : 997

Loaded artefacts  : dataset/train_preprocessing_artefacts.npz
  Training items  : 989
  Feature columns : 31

Test pivot shape  : (860, 989)  (users x training_items)
  Note: 18 interactions involve items not seen in training -> mapped to 0
Handcrafted features (test): 27 columns
cosine_sim_to_normal — test mean: 0.5037
mahal_dist — test mean: 0.8697
degsim_pearson — test mean: 0.4077
top5_corating_overlap — test mean: 0.5303

Total test features: 31
Column order matches training features. OK


In [10]:
# ==========================================================================
# Apply TRAINING StandardScaler to test features and export
#
# The scaler is restored from saved mean/scale rather than refitted.
# Refitting on test data would shift the feature distributions and make
# test scores incomparable to what the model learned.
# ==========================================================================
from sklearn.preprocessing import StandardScaler
import os

test_scaler             = StandardScaler()
test_scaler.mean_       = scaler_mean
test_scaler.scale_      = scaler_scale
test_scaler.var_        = scaler_scale ** 2
test_scaler.n_features_in_ = len(feature_cols)

test_out = agg_test.copy()
test_out[feature_cols] = test_scaler.transform(test_out[feature_cols])

# No "class" column — this is unlabelled inference data
os.makedirs("dataset", exist_ok=True)
test_out.to_csv(OUTPUT_CSV, index=False)

print(f"Saved: {OUTPUT_CSV}")
print(f"  Shape    : {test_out.shape}")
print(f"  Users    : {len(test_out)}")
print(f"  Features : {test_out.shape[1]}")
print(f"\nRun inference with:")
print(f"  python devnet.py --mode infer \\")
print(f"    --model_path ./model/devnet_user_features_with_labels_0.02cr_512bs_200ko_2d.h5 \\")
print(f"    --test_path  {OUTPUT_CSV} \\")
print(f"    --output_npz ./submission_devnet.npz \\")
print(f"    --network_depth 2")

Saved: dataset/test_features.csv
  Shape    : (860, 31)
  Users    : 860
  Features : 31

Run inference with:
  python devnet.py --mode infer \
    --model_path ./model/devnet_user_features_with_labels_0.02cr_512bs_200ko_2d.h5 \
    --test_path  dataset/test_features.csv \
    --output_npz ./submission_devnet.npz \
    --network_depth 2


c:\Users\xavie\Y3S1\CS421\project\cs421pain\proj_venv\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
